# Backpropagation: from the chain rule to a learned waveform

A neural network is a function with adjustable numbers called **parameters**.
It makes a prediction from an input. A **loss** is a number that measures how far
that prediction is from the supplied answer, or **target**. Training adjusts the
parameters to reduce the loss. Backpropagation uses the chain rule to calculate
the loss derivatives needed for those adjustments.

We will write these calculations ourselves, then ask whether a small loss means
we have recovered the physical signal.

### How to Proceed:

This notebook is intended to be run linearly, 1.1 --> 1.2 --> ..., --> 2 --> 3 --> ...

Any code expected for you to complete will contain `## FINISH_ME ##` indicating the code isn't expected to run until you have completed it.

### Contents:

| Part | The question we investigate |
|---|---|
| 1. Build and train | How does the chain rule become working code? |
| 2. Predict beyond the data | Has the network learned periodicity? |
| 3. Fill gaps | How can an already trained model supply missing values? |
| 4. Fit noisy observations | Can a model recover the average waveform? |
| 5. Train for too long | Can the loss improve while the waveform gets worse? |
| 6. Fit a physical model | What changes when we fit only the sine frequency? |

Run the cells in order using **Python (NumPy)**. The long run in Part 5 may take
a few minutes. We use NumPy for the calculations and Matplotlib for figures.

Each training step uses all the supplied observations. We prepare their positions
and any noise before training starts. We save copies of the model at the start,
25%, 50% and 100% of each run, so we can inspect how its predictions change.

Each experiment follows the same path: **question → prepare data → configure or
reuse a model → train or predict → inspect results → discuss**.

Our naming convention is `x` for input coordinates and `y` for displacement
values. The model receives the same coordinates that appear on the plots.
Observations occupy rows; each scalar input or target occupies one column. Model dictionaries hold adjustable
parameters; snapshots hold copies of a model and its predictions.

In waveform plots, black dashed lines show the truth, blue dots show
observations, orange lines show MLP predictions, and red crosses show filled
gaps. Grey marks an earlier model; green identifies the physical sinusoid fit.


## Part 1 — Building backpropagation, one step at a time

### 1.1 Prepare a simple problem

We want a multilayer perceptron (MLP) to approximate $y=\sin (x)$ on $0\leq x\leq2\pi$.
Each observation consists of a coordinate $x$ and a displacement $y$. We can sample
100 such pairs. The network will receive the coordinates and target values,
but its own prediction code will not contain the sine formula.

The network receives $x$ directly, from $0$ to $2\pi$, and learns to predict
$y=\sin (x)$. Both coordinates and displacements are dimensionless in this example.


**Exercise E1 — Prepare the observations**

Construct the input coordinates and target values. `np` is our short
name for NumPy. `linspace` creates equally spaced values, and
`np.column_stack([values])` places one array of values into a single column.
We write $N$ for the number of observations; a column has shape `(N, 1)`.
An array's `.shape` reports its row and column counts.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

NUMBER_OF_TRAINING_POINTS = 100

# These settings only control the appearance of the plots.
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11


In [ ]:
# Generate the training data for the sine function.
x_values = np.linspace( ## FINISH_ME ## )                             # Generate 100 points from 0 to 2π.
y_values = np.sin( ## FINISH_ME ## )                                  # Compute the sine of each x value.

# Store one observation per row, with one column for each scalar quantity.
x_train = np.column_stack([x_values])
y_train = np.column_stack([y_values])

# The training data is now stored in two arrays: inputs and targets.
print("Inputs, x_train: ", x_train.shape)
print("Targets, y_train:", y_train.shape)

print("First three (x, y) rows:")
# Python counts from zero. [:3, 0] selects the first three rows of column zero.
# column_stack places the input and target columns side by side for display.
print(np.column_stack((x_train[:3, 0], ## FINISH_ME ## )))


### 1.2 Initialise a tiny model, then apply the chain rule

Set the sine dataset aside for a moment. We will use **one input, one hidden
neuron and one output** to see every calculation. A hidden neuron is an
intermediate calculation between the input and the prediction.

**Step 1 — Choose the data and initialise the parameters.**

For this separate arithmetic example, choose an input $x=0.4$ and a target
$y=0.5$. These are convenient test values, not a pair taken from the sine dataset.
They stay fixed while we calculate derivatives.

Next, choose starting values for four model parameters. **Initialise** simply
means assign these values before using the model:

| Parameter | Role | Starting value |
|---|---|---|
| $w$ | Weight multiplying the input | 0.8 |
| $b$ | Bias added at the hidden neuron | 0.1 |
| $v$ | Weight multiplying the hidden neuron's output | −0.7 |
| $c$ | Bias added at the final output | 0.05 |

A **weight** multiplies an incoming value; a **bias** adds an offset. These four
numbers are the quantities we could change during training. Here we choose them
by hand, with no random initialisation. We will calculate a prediction and its
loss derivatives, but will not update the parameters yet.


In [ ]:
# Supplied data: one input and the answer we want the model to predict.
x = 0.4
target = 0.5

# Model parameters: assign starting values BEFORE evaluating the model.
w = 0.8    # Input -> hidden-neuron weight.
b = 0.1    # Hidden-neuron bias.
v = -0.7   # Hidden neuron -> output weight.
c = 0.05   # Output bias.

print("Fixed data: input =", x, "; target =", ## FINISH_ME ## )
print("Initial parameters: w =", w, "; b =", b, "; v =", v, "; c =", c)


**Step 2 — Calculate the prediction and loss: the forward pass.**

Now use the initial values, in this order:

$$z=wx+b,\qquad a=\tanh (z),\qquad \hat y=va+c,\qquad
L=\tfrac12(\hat y-y)^2.$$

The weighted sum $z$ is called the **pre-activation**. Applying $\tanh$ gives the
**activation** $a$, the hidden neuron's output. This activation function makes the
model nonlinear. The final output $\hat y$ uses a weight and bias with no further
activation function; we call this a linear output layer.

The loss $L$ is half the squared difference between prediction and target.
The factor one half makes its derivative simpler. The values $z$, $a$, $\hat y$
and $L$ are **calculated from** the parameters; they are not extra parameters
that we initialise independently.


In [ ]:
# Use the fixed data and the four parameters from the previous cell.
z = ## FINISH_ME ##  # Compute the hidden neuron input
a = ## FINISH_ME ##  # Compute the hidden neuron output

# The output of the hidden neuron is a, which is used to compute the final prediction.
prediction = ## FINISH_ME ##                       # Compute the model's prediction
scalar_loss = 0.5 * (prediction - target) ** 2     # ** 2 means squared.

# An f-string inserts values into text; :.4f displays four decimal places.
print(f"Hidden neuron: z = {z:.4f}, a = {a:.4f}")
print(f"Output: prediction = {prediction:.4f}, target = {target:.4f}")
print(f"Loss = {scalar_loss:.4f}")


**Step 3 — Calculate the loss derivatives: the backward pass.**

For any quantity $q$, $\partial L/\partial q$ tells us how the loss would change
for a small change in $q$, with other independent quantities held fixed.
Collecting the derivatives for all parameters gives the **gradient**.

Start with the loss derivative at the prediction:

$$\frac{\partial L}{\partial\hat y}=\hat y-y.$$

Then apply the chain rule one operation at a time. At the output,
$\partial\hat y/\partial a=v$. At the hidden activation,
$\partial a/\partial z=1-\tanh(z)^2 =1-a^2$. Therefore,

$$\frac{\partial L}{\partial a}=\frac{\partial L}{\partial\hat y}\,v,
\qquad
\frac{\partial L}{\partial z}=\frac{\partial L}{\partial a}(1-a^2).$$

For each parameter, multiply by its own local derivative:

$$\frac{\partial L}{\partial v}=\frac{\partial L}{\partial\hat y}\,a,
\quad \frac{\partial L}{\partial c}=\frac{\partial L}{\partial\hat y},
\quad \frac{\partial L}{\partial w}=\frac{\partial L}{\partial z}\,x,
\quad \frac{\partial L}{\partial b}=\frac{\partial L}{\partial z}.$$

For example, the last two expressions use $\partial z/\partial w=x$ and
$\partial z/\partial b=1$. In the code, `dloss_dw` means $\partial L/\partial w$.
A **sensitivity** is simply another name for one of these derivatives.


In [ ]:
# Start at the loss and differentiate the output calculation.
dloss_dprediction = prediction - target
dloss_dv = dloss_dprediction * a
dloss_dc = dloss_dprediction * 1.0  # The derivative of v*a + c with respect to c is 1.

# Continue through the hidden activation to its weighted sum.
dloss_da = ## FINISH_ME ##  # Differentiate the output with respect to the hidden-neuron output.
dloss_dz = ## FINISH_ME ##  # Differentiate the hidden-neuron output with respect to its input.

# Differentiate z = w*x + b with respect to each hidden-neuron parameter.
dloss_dw = dloss_dz * x
dloss_db = dloss_dz * 1.0

# We can also differentiate with respect to the input, although it stays fixed.
dloss_dx = dloss_dz * w

print(f"dL/dprediction = {dloss_dprediction:.4f}")
print(f"dL/da = {dloss_da:.4f}, dL/dz = {dloss_dz:.4f}")
print(f"Parameter derivatives: dL/dw = {dloss_dw:.4f}, dL/db = {dloss_db:.4f}")
print(f"                       dL/dv = {dloss_dv:.4f}, dL/dc = {dloss_dc:.4f}")

# No assignment to w, b, v or c occurs here: we have calculated derivatives only.


**Question:**

Why does the gradient not pass back through a bias?


**Answer:**

` ## FINISH_ME ## `


### 1.3 Assemble the full model as a dictionary

Return to the sine dataset from Section 1.1. We now build a larger network with
**one input, two hidden layers and one output**. A layer is a group of neurons
at the same stage of the calculation. Each hidden layer will have $H=10$ neurons;
$N=100$ is the number of observations. These are different counts.

Rows represent observations; columns represent neurons. The same model
parameters are used for every observation:

| Quantity | Shape | Meaning |
|---|---|---|
| `inputs` | `(N, 1)` | One coordinate per observation |
| `W1`, `b1` | `(1, H)`, `(1, H)` | First hidden layer parameters |
| `W2`, `b2` | `(H, H)`, `(1, H)` | Second hidden layer parameters |
| `W3`, `b3` | `(H, 1)`, `(1, 1)` | Output parameters |
| `a1`, `a2` | `(N, H)` | Hidden activations, calculated in the forward pass |
| `predictions` | `(N, 1)` | One predicted displacement per observation |

**Reading the NumPy code:**

`@` performs matrix multiplication; `*` multiplies
entries individually; `.T` transposes an array. Adding a bias row to an `(N, H)`
array repeats that row across the observations, called **broadcasting**.
`np.sum(..., axis=0, keepdims=True)` sums down the rows and keeps a row-shaped
result. Biases are rows because they supply one offset per neuron, shared
across observations. We will use these operations in the forward and backward passes.


**Exercise E2 — Assemble the model dictionary**

We collect all three layers’ weights
and biases in a Python dictionary called `model`. A dictionary associates each
name, called a **key**, with a value:

- `model["W1"]`, `model["W2"]`, `model["W3"]` hold the weight arrays.
- `model["b1"]`, `model["b2"]`, `model["b3"]` hold the bias arrays.

Together these six arrays contain all **141 adjustable numbers** for this network.
The dictionary stores the model’s complete parameter state. Section 1.4 will
define how to use it to calculate predictions: `forward_pass(inputs, model)`.
The data, predictions and gradients are separate from this stored state.

`initialize_parameters` builds and returns a new dictionary each time we call it.
`def` defines the function; `return` hands its result to the caller.
The assignment `model = initialize_parameters(HIDDEN_SIZE, MODEL_SEED)` creates
the model that we will inspect and pass between the later functions.

Here we use random initial weights instead of choosing each number by hand.
`generator.normal` draws numbers from a normal distribution with the given mean,
standard deviation and array shape. The scales below keep the weights moderate.
A **seed** fixes the random sequence so a run can be repeated. Biases start at
zero; different weights allow different neurons to develop different responses.


In [ ]:
# Choose the size and random starting point of the full network here.
HIDDEN_SIZE = 10
MODEL_SEED = 12345

def initialize_parameters(hidden_size, seed):
    """Return one model dictionary containing all weights and biases as float64 arrays."""

    # Use a random number generator with a fixed seed for reproducibility.
    generator = np.random.default_rng(seed)
    input_size = 1
    output_size = 1

    # Choose moderate random-weight scales based on the connected layer sizes.
    scale_1 = np.sqrt(2.0 / (input_size + hidden_size))
    scale_2 = np.sqrt(2.0 / (hidden_size + hidden_size))
    scale_3 = np.sqrt(2.0 / (hidden_size + output_size))

    # Assemble the complete parameter state: one weight array and bias per layer.
    model = {}

    # We need to make sure that the weights are initialized with a scale that is appropriate to the model being trained.
    # This helps prevent the vanishing or exploding gradient problem during training.
    # In this case, we are using a normal distribution with a mean of 0 and a standard deviation of the chosen scale, which is suitable for layers with ReLU activations.
    # We initialize the biases to zero, which is a common practice.

    # Layer 1: input -> hidden
    model["W1"] = generator.normal(0.0, scale_1, (input_size, hidden_size))
    model["b1"] = np.zeros((1, hidden_size))

    # Layer 2: hidden -> hidden
    # The hidden layer is fully connected to itself, so the weight matrix is square.
    model["W2"] = generator.normal( ## FINISH_ME ## )  # Use the appropriate scale and shape for the hidden-to-hidden weights
    model["b2"] = np.zeros( ## FINISH_ME ## )  # Use the appropriate shape for the hidden biases

    # Layer 3: hidden -> output
    model["W3"] = generator.normal(0.0, scale_3, (hidden_size, output_size))
    model["b3"] = np.zeros((1, output_size))

    return model

# Initialise the model parameters and print their shapes and total count.
model = initialize_parameters(HIDDEN_SIZE, MODEL_SEED)
print("Model container:", type(model).__name__)

# Print the shapes of all parameters and count the total number of scalar parameters.
parameter_names = ["W1", "b1", "W2", "b2", "W3", "b3"]
parameter_count = 0
for name in parameter_names:
    ## .shape gives the dimensions of the array, e.g. (1, 10) for a row vector of length 10.
    print(name, ## FINISH_ME ## )  # Print the name and shape of each parameter array.

    # .size is the number of scalar entries in this parameter array.
    parameter_count = parameter_count + model[name].size

print("Total scalar parameters:", parameter_count)


**Question:**

How many parameters does the network contain? Why use nonlinear hidden activations?


**Answer:**

` ## FINISH_ME ##`


### 1.4 Implement the forward pass

Use capital letters for arrays: $X$ contains the input coordinates, $Z$ the
pre-activations, $A$ the activations, and $\hat Y$ the predictions. Subscripts
identify the layer. The scalar calculation from Section 1.2 becomes

$$Z_1=XW_1+b_1,\quad A_1=\tanh (Z_1),\qquad
Z_2=A_1W_2+b_2,\quad A_2=\tanh (Z_2),\qquad
\hat Y=A_2W_3+b_3.$$

Pass the `model` dictionary from Section 1.3 into this function. Each layer
reads its own weights and biases from that same object; it does not initialise
new parameters.

Bias rows are broadcast over observations. `tanh` is applied to each entry.
The `cache` is a dictionary of intermediate values saved for the backward pass.
The function returns two results, `predictions` and `cache`; the call below gives
each a separate name.


**Exercise E3 — Implement the forward pass**

Translate these equations into array operations. Read the shape
comments to follow how the dimensions fit together.


In [ ]:
def forward_pass(inputs, model):
    """Evaluate the supplied scalar inputs; return predictions and intermediates."""

    # Layer 1: (N, 1) @ (1, H) + (1, H) -> (N, H).
    z1 = inputs @ model["W1"] + model["b1"]
    a1 = np.tanh(z1)

    # Layer 2: (N, H) @ (H, H) + (1, H) -> (N, H).
    z2 = ## FINISH_ME ##  # Compute the second hidden layer's weighted sum
    a2 = ## FINISH_ME ##  # Compute the second hidden layer's activation

    # Output: (N, H) @ (H, 1) + (1, 1) -> (N, 1).
    predictions = a2 @ model["W3"] + model["b3"]

    # Store the intermediate values in a cache for later use in backpropagation.
    cache = {"inputs": inputs, "z1": z1, "a1": a1, "z2": z2, "a2": a2}

    return predictions, cache


predictions_before, initial_cache = forward_pass(x_train, model)
print("First hidden activation shape:", initial_cache["a1"].shape)  # Print the shape of the first hidden activation
print("Second hidden activation shape:", ## FINISH_ME ## )          # Print the shape of the second hidden activation
print("Prediction shape:", predictions_before.shape)


### 1.5 Turn the prediction error into a gradient

The scalar example used one observation. Now average the squared errors over
all $N$ observations. Write $y_n$ for target $n$ and $\hat y_n$ for its prediction.
Their difference, prediction minus target, is the **residual**. We use the
**half mean squared error**:

$$L=\frac{1}{2N}\sum_{n=1}^N(\hat y_n-y_n)^2,
\qquad \frac{\partial L}{\partial\hat y_n}=\frac{\hat y_n-y_n}{N}.$$

The loss is one number; its derivative with respect to the predictions is an
`(N, 1)` array. The factor one half cancels the factor two from differentiation.
Dividing by $N$ makes this an average rather than a sum. We include that division
**once**, here, and must not divide again at every layer.

For plots we also use **root mean squared error (RMSE)**, $\sqrt{2L}$, which has
the same units as the output. Both describe agreement with whichever targets supplied the loss.
Neither automatically measures agreement with the underlying physical signal.


**Exercise E4 — Calculate loss and prediction derivatives**

Calculate the loss and its derivative. `np.mean` takes the
arithmetic average of the array entries.


In [ ]:
def loss_and_gradient(predictions, targets):
    """Half mean squared error and dL/d(predictions), for one output per sample."""

    # Results are returned as a float and a column vector of the same shape as predictions.
    number_of_points = targets.shape[0]
    residuals = predictions - targets

    # The half mean squared error is the average of the squared residuals, divided by two.
    loss = ## FINISH_ME ##  # Compute the half mean squared error

    # Each prediction has its own derivative: its residual divided by the observation count.
    gradient_predictions = residuals / number_of_points

    return float(loss), gradient_predictions


**RMSE helper — report errors in displacement units.**

`calculate_rmse(values, reference)` measures the size of the differences from
its named reference. We will reuse it for both observations and true waveforms.
It reports an error; it does not calculate gradients or update a model.


In [ ]:
def calculate_rmse(values, reference):
    """Measure the typical size of the differences from a supplied reference."""

    differences = values - reference
    mean_squared_difference = ## FINISH_ME ##  # Compute the mean of the squared differences

    return np.sqrt(mean_squared_difference)


# Evaluate the initial model against the training data.
initial_loss, initial_prediction_gradient = loss_and_gradient(predictions_before, y_train)
# Calculate the root mean squared error of the initial model.
initial_rmse = calculate_rmse(predictions_before, y_train)

print(f"Initial half mean squared error: {initial_loss:.6f}")
print(f"Initial RMSE against training targets: {initial_rmse:.6f}")


### 1.6 Carry the sensitivities backwards

The chain rule from Section 1.2 now acts on all observations and neurons
together. Let $Y$ contain the targets. Write $\Delta_\ell$ for the array of loss
derivatives with respect to the pre-activations in layer $\ell$; the code calls
these arrays `delta_1`, `delta_2` and `delta_3`. Start at the linear output,
then work towards the input:

$$\Delta_3=(\hat Y-Y)/N,$$
$$\Delta_2=(\Delta_3W_3^{\mathsf T})\odot(1-A_2^2),\qquad
\Delta_1=(\Delta_2W_2^{\mathsf T})\odot(1-A_1^2).$$

Here $\odot$ means entry-by-entry multiplication, and $A^2$ means squaring each
entry. Multiplication by the transposed weight matrix adds the contributions
from the neurons in the next layer. The factor $(1-A^2)$ differentiates `tanh`,
just as it did for one neuron.

For any layer with incoming activations $A$ and sensitivity $\Delta$,

$$\frac{\partial L}{\partial W}=A^{\mathsf T}\Delta,
\qquad \frac{\partial L}{\partial b}=\sum_{n=1}^{N}\Delta_{n,:}.$$

Why the matrix product? Weight $W_{ij}$ contributes $A_{ni}W_{ij}$ to neuron $j$
for observation $n$. Its gradient sums $A_{ni}\Delta_{nj}$ over observations.
A shared bias has local derivative one, so its gradient just sums the arriving
sensitivities. Each sample's contribution already contains the factor $1/N$.


**Exercise E5 — Implement backpropagation**

Implement these operations from the output layer backwards.
This function calculates gradients; it does not update parameters.


In [ ]:
def backward_pass(gradient_predictions, cache, model):
    """Apply the chain rule to the cached forward pass, without changing parameters."""
    inputs = cache["inputs"]
    a1 = cache["a1"]
    a2 = cache["a2"]
    gradients = {}

    # Output layer: its activation is the identity, whose derivative is 1.
    delta_3 = gradient_predictions                        # (N, 1)
    gradients["W3"] = a2.T @ delta_3                      # (H, 1)
    gradients["b3"] = np.sum(delta_3, axis=0, keepdims=True)

    # Sensitivity to A2, followed by the elementwise derivative of tanh(Z2).
    gradient_a2 = delta_3 @ model["W3"].T                 # (N, H)
    delta_2 = gradient_a2 * (1.0 - a2 ** 2)               # (N, H)
    gradients["W2"] = a1.T @ delta_2                      # (H, H)
    gradients["b2"] = np.sum(delta_2, axis=0, keepdims=True)

    # Repeat the same chain rule back through the first hidden layer.
    gradient_a1 =  ## FINISH_ME ##     # Differentiate the loss with respect to the first hidden layer's activation
    delta_1 =  ## FINISH_ME ##         # Differentiate the first hidden layer's activation with respect to its weighted sum
    gradients["W1"] = ## FINISH_ME ##  # Differentiate the loss with respect to the first hidden layer's weights
    gradients["b1"] = ## FINISH_ME ##  # Differentiate the loss with respect to the first hidden layer's biases

    return gradients

## Compute the gradients of the loss with respect to all parameters, using the cached forward pass.
initial_gradients = backward_pass(initial_prediction_gradient, initial_cache, model)

# Print the shapes of all gradients and count the total number of scalar gradients.
for name in parameter_names:
    print(name, "parameter and gradient shapes:", model[name].shape, initial_gradients[name].shape)


**Question:**

Why must the backward pass finish before we update any weights? Does backpropagation itself learn the parameters?


**Answer:**

` ## FINISH_ME ## `


### 1.7 Update the parameters

**Gradient descent** uses the derivatives to take a small step that aims to reduce
loss. Let $\theta$ collect the model parameters, $\nabla_\theta L$ their loss
derivatives, $k$ the update number, and $\eta$ the **learning rate**:

$$\theta_{k+1}=\theta_k-\eta\nabla_\theta L(\theta_k).$$

- A positive derivative tells us to **decrease** that parameter.
- A negative derivative tells us to **increase** it.
- Too large a step can overshoot; the minus sign does not guarantee improvement.


**Exercise E6 — Update the parameters**

Implement the six parameter updates below.
Calculate all gradients before changing any parameter.

This completes the functions you need to build. Section 1.8 supplies the loop
that calls them repeatedly to train the network.


In [ ]:
def update_parameters(model, gradients, learning_rate):
    """Update the weights and biases in the supplied model dictionary."""

    # Update each parameter by subtracting the learning rate multiplied by its gradient.
    model["W1"] = model["W1"] - learning_rate * gradients["W1"]
    model["b1"] = model["b1"] - learning_rate * gradients["b1"]
    model["W2"] = ## FINISH_ME ##  # Update the second layer weights using the learning rate and its gradient
    model["b2"] = ## FINISH_ME ##  # Update the second layer biases using the learning rate and its gradient
    model["W3"] = model["W3"] - learning_rate * gradients["W3"]
    model["b3"] = model["b3"] - learning_rate * gradients["b3"]


### 1.8 Training — put the pieces together

**You have built the scaffold.** Each earlier function does one part of learning;
now we connect them and repeat the process.

> **Supplied code — read and run it.** You do not need to write this training
> loop. Keep it intact. Your task is to understand how
> it uses the functions you have built.

#### Find your functions in the complete calculation

| Function | What it contributes |
|---|---|
| `initialize_parameters` — 1.3 | Assemble the `model` dictionary of weights and biases, **once before training** |
| `forward_pass` — 1.4 | Use the current parameters to produce predictions and a `cache` |
| `loss_and_gradient` — 1.5 | Compare with targets; return loss and its prediction derivatives |
| `backward_pass` — 1.6 | Use those derivatives and the cache to calculate parameter gradients |
| `update_parameters` — 1.7 | Change weights and biases using the learning rate |

#### Follow one trip around the loop

1. **Predict:** run the current network forwards. Save its intermediate values in `cache`.
2. **Compare:** calculate the loss and `gradient_predictions` from the fixed targets.
3. **Differentiate:** pass both the derivatives and cache into `backward_pass`.
4. **Update:** use the resulting gradients to change the parameters.

Then return to step 1. The new weights need a **new forward pass and cache**;
we cannot reuse intermediate values from the old model.

**Fixed throughout:** input coordinates, targets and learning rate.
**Changed each update:** weights and biases. Predictions, loss and gradients
are then recalculated. Each **training update** uses all observations. Batching and the term
“epoch” will be introduced together in the later notebook.


#### First, make one learning step easy to follow

`train_one_step` calls the four functions you built earlier, in order:
**predict → compare → differentiate → update**.

It changes the supplied `model` dictionary and returns the loss measured
**before that update**. The next call will use the new weights and biases.


In [ ]:
def train_one_step(inputs, targets, model, learning_rate):
    """Make one parameter update using all observations; return its starting loss."""

    # 1. PREDICT: use the current weights and save the intermediate values.
    predictions, cache = forward_pass(inputs, model)

    # 2. COMPARE: measure the error and its sensitivity to the predictions.
    loss, gradient_predictions = loss_and_gradient(predictions, targets)

    # 3. DIFFERENTIATE: calculate all parameter gradients before changing anything.
    gradients = backward_pass(gradient_predictions, cache, model)

    # 4. UPDATE: change the weights and biases in the model dictionary.
    update_parameters(model=model, gradients=gradients, learning_rate=learning_rate)

    return loss


#### Keep a record without changing the learning

`capture_snapshot` looks at the model **as it is now**. It calculates predictions
and loss, copies the parameter arrays, and prints the training progress.
It never calls `backward_pass` or `update_parameters`. The name `_` marks a
returned value that is not needed at this time.

- `snapshot["model"]` holds the copied weight and bias arrays.
- `.copy()` prevents later training from changing the saved arrays.
- `prediction_inputs` supplies extra coordinates for drawing the curve. These
  coordinates are not used for parameter updates.
- The returned dictionary contains one saved state, called a **snapshot**.

We save **0%, 25%, 50% and 100%**. The first snapshot is taken before any updates;
the last is taken after all updates. `//` means integer division: for example,
`15000 // 4` gives the quarter-way checkpoint at 3,750 updates.


In [ ]:
def capture_snapshot(
    inputs,
    targets,
    model,
    prediction_inputs,
    completed_updates,
    number_of_updates,
):
    """Record the current model and report progress, without updating its parameters."""

    # Measure this saved model on the training observations.
    predictions, _ = forward_pass(inputs, model)
    loss, _ = loss_and_gradient(predictions, targets)

    # Copy every parameter array so this state survives later training updates.
    saved_model = {}
    for name in model:
        saved_model[name] = model[name].copy()

    # Evaluate extra coordinates only to draw the saved waveform.
    curve_predictions, _ = forward_pass(prediction_inputs, model)

    snapshot = {
        "updates": completed_updates,
        "model": saved_model,
        "training_inputs": inputs.copy(),
        "training_predictions": predictions.copy(),
        "curve_predictions": curve_predictions.copy(),
        "loss": float(loss),
    }

    # Report the progress of training in a single line of text.
    percentage = 100 * completed_updates // number_of_updates
    print(
        f"Training {percentage:3d}% | "
        f"updates {completed_updates:6d} | loss {loss:.6e}"
    )

    return snapshot


#### Read the complete training loop

The loop now has two jobs: **save a checkpoint when needed**, then **take one
learning step**. The functions above contain the details of each job.

`completed_updates` starts at zero and counts how many updates have already
happened. We save a checkpoint before the next update, so its parameters,
predictions and loss all describe the same state.

`loss_history` and `model_snapshots` are lists. `.append(...)` adds one item.
After the loop, we record the final state separately: this measures the result
of the last update without making another update. That is why the history has
one more loss value than the number of updates.

In the longer calls, `inputs=inputs` means “pass our `inputs` array to the
argument named `inputs`”. Naming the arguments makes each value’s role visible.


In [ ]:
def train_network(
    inputs,
    targets,
    model,
    number_of_updates,
    learning_rate,
    prediction_inputs,
):
    """Train the supplied model; return its loss history and four saved states."""

    # Save the starting state and the states one quarter and halfway through training.
    checkpoint_updates = [0, number_of_updates // 4, number_of_updates // 2]

    loss_history = []
    model_snapshots = []

    # Each trip around this loop makes exactly one parameter update.
    for completed_updates in range(number_of_updates):

        # Save the current model at 0%, 25% and 50%, before changing its weights.
        if completed_updates in checkpoint_updates:
            snapshot = capture_snapshot(
                inputs=inputs,
                targets=targets,
                model=model,
                prediction_inputs=prediction_inputs,
                completed_updates=completed_updates,
                number_of_updates=number_of_updates,
            )

            model_snapshots.append(snapshot)

        # Learn from all observations once, then store the loss from that step.
        loss = train_one_step(
            inputs=inputs,
            targets=targets,
            model=model,
            learning_rate=learning_rate,
        )

        loss_history.append(loss)

    # Training is finished. Measure and save the 100% state without another update.
    final_snapshot = capture_snapshot(
        inputs=inputs,
        targets=targets,
        model=model,
        prediction_inputs=prediction_inputs,
        completed_updates=number_of_updates,
        number_of_updates=number_of_updates,
    )

    # Save the final state and its loss, then return the complete history of losses and snapshots.
    model_snapshots.append(final_snapshot)
    loss_history.append(final_snapshot["loss"])

    return np.array(loss_history), model_snapshots


**Question:**

Suppose you run `train_network` for 1,000 updates with `learning_rate=0`.
The loop still calculates gradients and saves snapshots at 0%, 25%, 50% and 100%.

- **Has the model learned anything?**
- Explain what you expect to happen to its weights and biases, predictions and loss.
- Does saving a snapshot change your answer?


**Answer:**

` ## FINISH_ME ## `

### 1.9 Train the clean-waveform model

#### Connect the data and run it

The next cell chooses the settings, creates plotting coordinates, initialises
one model, and passes everything to `train_network`.

| Argument | Value in this experiment |
|---|---|
| `inputs`, `targets` | `x_train`, `y_train`: the observations from Section 1.1 |
| `model` | The model dictionary containing all six weight and bias arrays |
| `number_of_updates` | 15,000 complete parameter updates |
| `learning_rate` | 0.05: the size of each gradient-descent step |
| `prediction_inputs` | `x_curve`: coordinates used only to draw saved curves |

The loop changes the supplied `model` dictionary, so it holds the trained
model afterwards. It returns **two records**: `loss_history` and `clean_snapshots`.
The history includes the initial loss and every updated state: 15,001 values.


In [ ]:
# Training choices: how many updates to make, and the size of each step.
NUMBER_OF_UPDATES = 15000
LEARNING_RATE = 0.05

# This dense grid is for plotting the fitted function alongside the true sine.
curve_values = np.linspace(0.0, 2.0 * np.pi, 100)
x_curve = np.column_stack([curve_values])
y_curve_true = np.sin(x_curve)


# Start from the same initial model each time this cell runs.
model = initialize_parameters( ## FINISH_ME ## )  # Use the same hidden size and seed as before

# Train the network on the input coordinates and targets, returning the loss history and snapshots of the model at key points.
loss_history, clean_snapshots = train_network(
    inputs=x_train,
    targets=y_train,
    model=model,
    number_of_updates=NUMBER_OF_UPDATES,
    learning_rate=LEARNING_RATE,
    prediction_inputs=x_curve,
)


#### Inspect the result

Read the training loss alongside the waveform. How well did the model describe
the sine before training, and how did its predictions change after training?
The loss uses the supplied training targets; the dashed curve is the true sine.


In [ ]:
# subplots returns the figure and its two plotting areas (axes).
# semilogy uses a logarithmic vertical scale so small losses remain visible.
# Index 0 selects the first snapshot; -1 selects the last.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

# arange creates update numbers 0, 1, ...; len counts the stored losses.
axes[0].semilogy(np.arange(len(loss_history)), loss_history, color="tab:orange")
axes[0].set(xlabel="Training updates", ylabel="Half MSE against training targets", title="The training loss decreases")

# Plot the true sine function and the model's predictions before and after training.
axes[1].plot(x_curve[:, 0], ## FINISH_ME ##, color="black", linestyle="--", label="True sine")

# Plot the model's predictions before training.
axes[1].plot(x_curve[:, 0], clean_snapshots[0]["curve_predictions"][:, 0], color="grey", label="Before training")

# The final snapshot of the model is stored in the last element of the list, which is accessed using -1 as the index.
# Plot the model's predictions after training.
axes[1].plot(x_curve[:, 0], ## FINISH_ME ##, color="tab:orange", label="After training")

# Set the labels and title for the second plot, and add a legend to identify the curves.
axes[1].set(xlabel="Coordinate x", ylabel="Displacement", title="The learned function")
axes[1].legend()

# Display the two plots side by side.
plt.show()

clean_training_rmse = calculate_rmse(clean_snapshots[-1]["training_predictions"], y_train)
print(f"Final RMSE against training targets: {clean_training_rmse:.5f}")


**Question:**

Trace one training update.
Which function changes the parameters, and why do we need to run the model forwards again afterwards? What do the 2 plots show?


**Answer:**

` ## FINISH_ME ## `


## Part 2 — Predict beyond the training interval

Keep the trained parameters fixed and ask for predictions further along the
coordinate. Pass those coordinates directly to the model. The shaded region
contains the training data. Predicting outside that interval is called
**extrapolation**.



**Question for this experiment:** does a good fit over one period imply a periodic model?

**Model choice:** reuse the trained Part 1 `model`; its parameters stay fixed.


**Exercise E7 — Predict beyond the training interval**

Evaluate the existing model on a wider interval. Before running
this cell, predict whether the network will repeat the waveform.


### 2.1 Prepare coordinates and predict with the existing model


In [ ]:
# Create a dense grid of x values that extends well beyond the training interval.
extended_values = np.linspace( ## FINISH_ME ## )  # Generate 700 points from 0 to 10π.
x_extended = np.column_stack([extended_values])

# Evaluate the trained model on this extended grid to see how well it generalises.
y_extended, extended_cache = forward_pass(x_extended, model)


### 2.2 Inspect the predictions

The shaded interval contains the training observations. Compare the orange
prediction with the black dashed sine, especially beyond that interval.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.axvspan(0.0, 2.0 * np.pi, color="grey", alpha=0.15, label="Training interval")

# Plot the true sine function and the model's predictions over the extended range.
ax.plot(x_extended[:, 0], np.sin(x_extended[:, 0]), color="black", linestyle="--", label="True sine")

# Plot the model's predictions over the extended range.
ax.plot(x_extended[:, 0], ## FINISH_ME ##, color="tab:orange", label="Network")

# Set the labels and title for the plot, and add a legend to identify the curves.
ax.set(xlabel="Coordinate x", ylabel="Displacement", title="A good local fit does not imply periodicity")
ax.legend()

# Display the plot of the model's predictions over the extended range.
plt.show()


**Question:**

Why does the network fail to repeat the sine wave?


**Answer:**

` ## FINISH_ME ## `


## Part 3 — Fill gaps with the pre-trained model

Create a record with slightly irregular observation positions, then remove some
observations. We know each retained coordinate exactly, and its displacement
lies exactly on the sine. Uneven spacing does not itself introduce measurement
error in either coordinate or displacement.

**Exercise E8 — Fill gaps with the trained model**

Use the already trained Part 1 model to predict the missing
values. We do not train on this incomplete record. The four panels replay the
saved Part 1 states so we can see how the ability to fill gaps developed.


### 3.1 Prepare the uneven record

**Two small helpers prepare the record.** The first chooses uneven coordinates;
the second separates the observations we keep from the gaps.

`make_irregular_coordinates` starts with equally spaced positions and moves
each interior position by less than a quarter of the usual spacing. The first
and last positions stay fixed, and the positions remain in increasing order.

`np.column_stack([positions])` puts the position values into one column, giving
one row per observation. We create that column once; later row selections keep
the column shape needed by the model.


In [ ]:
def make_irregular_coordinates(number_of_points, generator):
    """Create a column of slightly uneven observation positions."""
    positions = np.linspace(0.0, 2.0 * np.pi, number_of_points)
    usual_spacing = positions[1] - positions[0]

    # There are two endpoints, so only the other positions receive offsets.
    number_of_interior_points = number_of_points - 2
    offsets = generator.uniform(-0.25, 0.25, size=number_of_interior_points)
    offsets = offsets * usual_spacing

    # [1:-1] selects every position except the first and last.
    positions[1:-1] = positions[1:-1] + offsets

    return np.column_stack([positions])


`split_observations` makes one keep/remove decision for each row. A **mask** is
an array of `True` and `False` values; using it to select rows keeps the `True`
ones. Applying the same mask to coordinates and values keeps each pair together.

Here each interior observation has a 45% chance of being removed. The actual
number removed can vary; we always retain both endpoints. The helper returns
four arrays: observed coordinates and values, then missing coordinates and
their true values. The missing true values are a reference for this simulated
example; they are not passed to the model when filling the gaps.


In [ ]:
def split_observations(coordinates, values, missing_fraction, generator):
    """Separate a record into retained observations and missing points."""
    random_choices = generator.random(len(coordinates))
    retained_mask = random_choices > missing_fraction

    # Keep both ends of the record visible.
    retained_mask[0] = True
    retained_mask[-1] = True
    missing_mask = ~retained_mask  # Reverse each True/False decision.

    # Select whole rows: each result still has one column.
    observed_x = coordinates[retained_mask]
    observed_y = values[retained_mask]
    missing_x = coordinates[missing_mask]
    missing_y = values[missing_mask]

    return observed_x, observed_y, missing_x, missing_y


### 3.2 Predict with the existing model

Use the same random generator for
both helpers so the example is repeatable. Generate the positions and keep/remove
decisions once. Then reuse the trained Part 1 model; no training takes place here.

Only the coordinates are uneven. Every value in this record lies on the true
sine, so this example contains no measurement noise.


In [ ]:
cadence_generator = np.random.default_rng(3101)

# 1. Create the uneven record: each coordinate has its matching sine value.
irregular_x = make_irregular_coordinates(number_of_points=100, generator=cadence_generator)
irregular_y = np.sin(irregular_x)

# 2. Keep some observations and set the missing points aside.
x_observed, y_observed, x_missing, y_missing_true = split_observations(
    coordinates=irregular_x,
    values=irregular_y,
    missing_fraction=0.45,
    generator=cadence_generator,
)

# 3. Pass the missing coordinates directly to the already-trained model.
y_filled, missing_cache = forward_pass( ## FINISH_ME ## )

print("Retained observations:", len(x_observed))
print("Missing observations to fill:", len(x_missing))


### 3.3 Inspect the saved predictions

**Build the figure from three small drawing helpers.** Parts 3–5 reuse this code.
Each helper has one job:

| Helper | What it does |
|---|---|
| `create_snapshot_panels` | Make four panels in reading order |
| `draw_saved_waveform` | Draw the true sine, observations and one saved prediction |
| `draw_gap_predictions` | Use a saved model to predict at missing coordinates |

`figure` is the whole picture; an `axis` is one plotting panel. The caller supplies the curve
coordinates and reference values from Part 1. `[:, 0]` selects column zero as a
one-dimensional array for plotting. These helpers draw and predict; they do not train.


In [ ]:
def create_snapshot_panels():
    """Create the figure and return its four panels in reading order."""
    figure, axes = plt.subplots(2, 2, figsize=(10, 6),
                                sharex=True, sharey=True, constrained_layout=True)

    # Shared axes give every panel the same coordinate and displacement scales.
    panels = [axes[0, 0],  # Top left.
              axes[0, 1],  # Top right.
              axes[1, 0],  # Bottom left.
              axes[1, 1],  # Bottom right.
             ]

    return figure, panels


In [ ]:
def draw_saved_waveform(axis, snapshot, observed_x, observed_y, curve_x, curve_y, percentage):
    """Draw the reference, observations and saved waveform on one panel."""

    # The curve coordinates and the saved predictions are extracted from the snapshot for plotting.
    curve_coordinates = curve_x[:, 0]
    saved_displacements = snapshot["curve_predictions"][:, 0]

    # Black dashed line: the known sine wave.
    axis.plot(curve_coordinates, curve_y[:, 0], color="black", linestyle="--", label="True sine")

    # Blue dots: the observations supplied for this figure.
    axis.scatter(observed_x[:, 0], ## FINISH_ME ##, s=15, color="tab:blue", alpha=0.7, label="Observations")

    # Orange line: predictions recorded when this model was saved.
    axis.plot(curve_coordinates, saved_displacements, color="tab:orange", label="Saved network")

    completed_updates = snapshot["updates"]
    axis.set_title(f"{percentage}% — {completed_updates:,} updates")
    axis.set_xlabel("Coordinate x")
    axis.set_ylabel("Displacement")


In [ ]:
def draw_gap_predictions(axis, snapshot, missing_x):
    """Predict at the missing coordinates and draw the results as red crosses."""
    saved_model = snapshot["model"]

    missing_predictions, _ = forward_pass(missing_x, ## FINISH_ME ## )  # Use the saved model to predict at the missing coordinates

    axis.scatter(missing_x[:, 0], missing_predictions[:, 0], marker="x", s=22, color="tab:red", label="Filled gaps")



**Now assemble the four-panel figure.** The main function creates the panels,
then draws one saved state in each: **0%, 25%, 50% and 100%** of training.

`missing_x=None` means there are no gap markers to draw. Part 3 supplies missing
coordinates directly; Parts 4 and 5 omit this argument.
`curve_x` and `curve_y` supply the reference curve explicitly. The final three lines add a shared legend,
a title and display the figure.


In [ ]:
def plot_training_snapshots(snapshots, observed_x, observed_y, curve_x, curve_y, title, missing_x=None):
    """Show the four saved models, with optional predictions in the gaps."""

    figure, panels = create_snapshot_panels()
    training_percentages = [0, 25, 50, 100]

    # Select one panel and the saved model that belongs in it.
    for panel_index in range(4):

        # Each panel shows the same observations and the same true sine curve, but a different saved model.
        axis = panels[panel_index]
        snapshot = snapshots[panel_index]
        percentage = training_percentages[panel_index]

        # Draw the reference sine curve, the retained observations, and the saved waveform for this snapshot.
        draw_saved_waveform(axis=axis, snapshot=snapshot, observed_x=observed_x, observed_y=## FINISH_ME ##,
                            curve_x=curve_x, curve_y=## FINISH_ME ##, percentage=percentage)

        # Gap predictions are optional; the waveform is always drawn.
        if missing_x is not None:
            draw_gap_predictions(axis=axis, snapshot=snapshot, missing_x=missing_x)

    panels[0].legend(fontsize=8)
    figure.suptitle(title)
    plt.show()


plot_training_snapshots(snapshots=clean_snapshots, observed_x=## FINISH_ME ##, observed_y=y_observed,
                        curve_x=## FINISH_ME ##, curve_y=y_curve_true,
                        title="Gap filling with the saved Part 1 models", missing_x=x_missing)


**Question:**

Do the retained observations alone determine these filled values?


**Answer:**

` ## FINISH_ME ## `


## Part 4 — Learn the average waveform from noisy observations

Imagine measuring the sine wave with an instrument that adds a small random
offset to each reading. Can the network learn the underlying waveform?

We will **make observations → train a fresh model → watch it learn → compare
with the true sine**. The short helpers below keep the details of each step in
one place. They reuse the model and training functions from Part 1.

**Exercise E9 — Train on noisy observations**

Train on the noisy observations. Look for a curve that follows
the main waveform without passing through every measured point.

### 4.1 Make one set of noisy observations

`make_noisy_observations` returns three arrays: coordinates, measured values,
and the noise offsets we added. The coordinates are evenly spaced; only the
measured displacement contains noise.

The offsets come from a normal distribution centred on zero. Their standard
deviation is **0.20**, compared with a sine amplitude of one. This describes the
noise scale, not a maximum offset. We create the random generator at the call site; its seed makes the example repeatable.
We generate these observations **once** and keep them fixed during training.


In [ ]:
def make_noisy_observations(number_of_points, noise_std, generator):
    """Create evenly spaced sine measurements with random displacement offsets."""

    # Generate a column of evenly spaced coordinates from 0 to 2π.
    coordinates = np.linspace(0.0, 2.0 * np.pi, number_of_points)
    coordinates = np.column_stack([coordinates])

    # Add normally distributed noise to the sine values to simulate measurement errors.
    offsets = generator.normal(0.0, noise_std, size=coordinates.shape)

    # The true sine values are computed, and the noisy measurements are generated by adding the offsets.
    true_values = np.sin(coordinates)
    measured_values = true_values + offsets

    return coordinates, measured_values, offsets


# Make the observations once, before starting any training.
measurement_sigma = 0.20
measurement_generator = np.random.default_rng(2027)

# Generate the noisy observations and their offsets.
x_noisy, y_noisy, noise_offsets = make_noisy_observations(number_of_points=100,
                                                          noise_std=measurement_sigma,
                                                          generator=measurement_generator)



### 4.2 Train a fresh model on those observations

`train_fresh_model` creates new weights and biases, then calls the training
loop we already know. The call below supplies the same network size, model seed,
learning rate and number of updates as Part 1. These settings are explicit
arguments, so the helper does not depend on hidden training settings.

It returns the trained model, its loss history and its four saved states.
We give these results names beginning with `noisy_` so the clean Part 1 model
stays available. Only the measured values `y_noisy` are supplied as targets;
the network is not given the clean sine as an answer.


In [ ]:
def train_fresh_model(
    inputs, targets, hidden_size, seed, number_of_updates, learning_rate, prediction_inputs
):
    """Initialise and train a new network using the settings supplied by the caller."""

    # Start from a new random model each time this function is called.
    fresh_model = initialize_parameters(hidden_size=hidden_size, seed=seed)

    # Train the network on the supplied inputs and targets, returning the loss history and snapshots of the model at key points.
    losses, snapshots = train_network(
        inputs=inputs,
        targets=targets,
        model=fresh_model,
        number_of_updates=number_of_updates,
        learning_rate=learning_rate,
        prediction_inputs=prediction_inputs,
    )

    return fresh_model, losses, snapshots



In [ ]:
# Train a new model on the noisy observations, using the same training settings as before.
noisy_model, noisy_loss_history, noisy_snapshots = train_fresh_model(
    inputs=x_noisy,
    targets=y_noisy,
    hidden_size= ## FINISH_ME ##,  # Use the same hidden size as before
    seed= ## FINISH_ME ##,         # Use the same seed as before
    number_of_updates=NUMBER_OF_UPDATES,
    learning_rate=LEARNING_RATE,
    prediction_inputs=x_curve,
)


### 4.3 Watch the learned waveform develop

Reuse the Part 3 plotting helper. The blue dots are the same noisy observations
in every panel; the orange line is the model saved at that stage of training.
The dashed black line shows the true sine for comparison.

Look for the overall waveform becoming clearer, even where the individual
measurements lie above or below it.


In [ ]:
plot_training_snapshots(
    snapshots=noisy_snapshots,
    observed_x=x_noisy,
    observed_y= ## FINISH_ME ##,  # Use the noisy observations as the y values
    curve_x=x_curve,
    curve_y= ## FINISH_ME ##,  # Use the true sine values as the y values
    title="Learning from noisy observations",
)


### 4.4 Compare the observations and the fitted waveform

The training loss measures agreement with the **noisy observations**.
Here we ask a different question: how close are the observations and the final
model to the **true sine**?

`calculate_rmse` reports the root mean squared error introduced in Part 1.
Smaller values mean closer agreement with the supplied reference, in the same
units as the displacement. We compare both sets of values at the same coordinates.
The clean reference is available because we made this simulated experiment;
these calculations do not update the model.


In [ ]:
# Use the last saved model: index -1 means the final entry in the list.
final_predictions = noisy_snapshots[-1]["training_predictions"]
true_values = np.sin(x_noisy)

raw_noise_rmse = calculate_rmse( ## FINISH_ME ##, true_values)  # Calculate RMSE for the raw noisy observations
fitted_signal_rmse = calculate_rmse( ## FINISH_ME ##, true_values)  # Calculate RMSE for the fitted model predictions

print(f"Raw observations: RMSE from the true sine = {raw_noise_rmse:.4f}")
print(f"Fitted waveform:  RMSE from the true sine = {fitted_signal_rmse:.4f}")


**Question:**

Why can the model recover a smoother waveform? Should we insist on zero training loss?


**Answer:**

` ## FINISH_ME ## `


## Part 5 — A lower loss can give a worse waveform

**Question for this experiment:** can fitting the observations more closely
make our estimate of the physical signal worse?

We use 16 randomly positioned sine observations with noise of standard deviation
**0.05**, a fresh model with **30 neurons per hidden layer**, and **600,000 updates**.
The observations remain fixed. We compare the saved models at 25% and 100% of
that same training run.

This deliberately flexible model can follow small details in the observations.
When fitting those details harms the inferred waveform, we call it **overfitting**.
This example shows what can happen; more training does not always cause it.

**Exercise E10 — Compare fitting the data with recovering the signal**

Run the experiment and explain whether the lowest training loss selects the
best waveform. Use the two error references in Section 5.5 to support your answer.


### 5.1 Prepare a sparse, noisy record

The helper chooses random positions, sorts them, and adds displacement noise.
As in Parts 3 and 4, it receives a random generator from the caller and returns
columns. Make this dataset once before training.


In [ ]:
def make_sparse_noisy_observations(number_of_points, noise_std, generator):
    """Create a small sine dataset with random positions and noisy values."""

    # Generate random positions uniformly distributed between 0 and 2π.
    positions = generator.uniform(0.0, 2.0 * np.pi, number_of_points)
    coordinates = np.column_stack( ## FINISH_ME ## )  # Convert the sorted positions into a column vector

    # Add normally distributed noise to the sine values to simulate measurement errors.
    offsets = generator.normal(0.0, noise_std, size=coordinates.shape)
    measured_values = np.sin(coordinates) + offsets

    return coordinates, measured_values, offsets

# Set the number of sparse points and the standard deviation of the noise for the sparse dataset.
number_of_sparse_points = 16
SPARSE_NOISE_STD = 0.05
sparse_generator = np.random.default_rng(501)

# Generate the sparse noisy observations and their offsets.
x_sparse, y_sparse, sparse_noise = make_sparse_noisy_observations(number_of_points=number_of_sparse_points,
                                                                  noise_std=SPARSE_NOISE_STD, generator=sparse_generator)

# This clean reference is for assessment only, not a training target.
y_sparse_true = np.sin(x_sparse)


### 5.2 Configure a fresh, flexible model

The helper starts a new MLP, then gives its first layer steeper transitions
spread across $0$ to $2\pi$, the same interval as the input coordinates.
This makes small details easier to represent.
All weights and biases remain trainable using the same backpropagation code.

`transition_centres` is a row of neuron settings, like the bias row in Part 1;
it is not a column of observations. `np.array([centres])` creates that single row.


In [ ]:
def initialize_flexible_model(hidden_size, seed, transition_slope):
    """Start a fresh MLP with steep first-layer transitions across the input interval."""

    # Use the same parameter initialization as before,
    # but then overwrite the first layer's weights and biases to create a steep transition function.
    fresh_model = initialize_parameters(hidden_size=hidden_size, seed=seed)

    # The centres of the tanh transitions are evenly spaced across the input interval [0, 2π].
    centres = np.linspace( ## FINISH_ME ## )  # Generate 'hidden_size' number of centres from 0 to 2π
    transition_centres = np.array([centres])

    # tanh(s * (x - centre)) uses weight s and bias -s * centre.
    fresh_model["W1"] = np.full((1, hidden_size), transition_slope)
    fresh_model["b1"] = -transition_slope * transition_centres

    return fresh_model

# The sparse model has a smaller hidden size and a steeper transition slope to fit the sparse data more closely.
SPARSE_HIDDEN_SIZE = 30
SPARSE_TRANSITION_SLOPE = 4.0
sparse_model = initialize_flexible_model(hidden_size=SPARSE_HIDDEN_SIZE,
                                         seed=MODEL_SEED, transition_slope=SPARSE_TRANSITION_SLOPE)


### 5.3 Train on the fixed observations

Only the noisy measurements are supplied as targets. Keep the network size,
learning rate and observations fixed while training continues from 25% to 100%.
The weights and biases keep changing; we compare checkpoints from the same run.


In [ ]:
SPARSE_UPDATES = 600000
SPARSE_LEARNING_RATE = 0.03

# Train the sparse model on the sparse noisy observations, returning the loss history and snapshots of the model at key points.
sparse_loss_history, sparse_snapshots = train_network(
    inputs=  ## FINISH_ME ##,  # Use the sparse x as the inputs
    targets= ## FINISH_ME ##,  # Use the sparse y as the targets
    model=sparse_model,
    number_of_updates=SPARSE_UPDATES,
    learning_rate=SPARSE_LEARNING_RATE,
    prediction_inputs=x_curve,
)


### 5.4 Inspect the saved waveforms

Use the same colours and four-panel display as Parts 3 and 4. Watch how the
orange prediction changes between the blue observations as training continues.


In [ ]:
plot_training_snapshots(
    snapshots=sparse_snapshots,
    observed_x=x_sparse,
    observed_y=y_sparse,
    curve_x= ## FINISH_ME ##,  # Use the dense x_curve as the x values for plotting
    curve_y= ## FINISH_ME ##,  # Use the dense y_curve_true as the y values for plotting
    title="A flexible network follows the noisy observations",
)


### 5.5 Compare errors against the data and the signal

**Read these two diagnostics together.** The left panel compares error against
the noisy training targets with error against the clean reference. Reference
errors are measured at only four checkpoints; the dashed line connects them.
The right panel shows prediction minus the true sine after 25% and 100% of
training. This is a residual from the physical signal, not from the noisy targets.
Grey edges lie outside the observed coordinate range, so distortions in the
unshaded region cannot be explained simply as extrapolation.


In [ ]:
def draw_training_and_signal_errors(axis, losses, saved_updates, signal_errors):
    """Compare agreement with noisy targets against agreement with the true sine."""
    # For the half-MSE loss, sqrt(2 * loss) is the corresponding RMSE.
    training_errors = np.sqrt(2.0 * losses)

    # Draw the training RMSE and the RMSE against the true sine on a logarithmic scale.
    axis.semilogy(np.arange(len(losses)), training_errors,
        color="tab:orange", label="RMSE against noisy targets")

    # Draw the RMSE against the true sine for the saved models, using green circles connected by dashed lines.
    axis.semilogy(saved_updates, signal_errors, "o--",
        color="tab:green", label="RMSE against true sine (dense grid)")

    # Set the labels, title, and legend for the plot.
    axis.set_xlabel("Training updates")
    axis.set_ylabel("RMSE")
    axis.set_title("Fitting the data versus recovering the signal")
    axis.legend(fontsize=8)


def draw_waveform_residuals(axis, observed_x, noise, curve_x, truth, snapshots):
    """Show measurement noise and the saved models' residuals from the true sine."""

    # Draw a horizontal line at zero to indicate perfect agreement with the true sine.
    axis.axhline(0.0, color="black", linewidth=1)

    # Shade the regions outside the training interval to highlight where the model was trained.
    axis.axvspan(curve_x[0, 0], observed_x[0, 0], color="grey", alpha=0.15)
    axis.axvspan(observed_x[-1, 0], curve_x[-1, 0], color="grey", alpha=0.15)
    axis.scatter(observed_x[:, 0], noise[:, 0], s=22, color="tab:blue", label="Measurement noise")

    # Grey marks the earlier model; orange marks the final MLP prediction.
    for index, label, colour in [(1, "25% trained", "grey"),
                                 (3, "100% trained", "tab:orange")]:

        residuals = snapshots[index]["curve_predictions"] - truth
        axis.plot(curve_x[:, 0], residuals[:, 0], color=colour, label=label)

    axis.set_xlabel("Coordinate x")
    axis.set_ylabel("Prediction minus true sine")
    axis.set_title("Extra training increases the distortion")
    axis.legend(fontsize=8)


In [ ]:
# The following code block calculates the RMSE of the model's predictions against the true sine values for each snapshot.
# Then plot the training loss and RMSE against the true sine, as well as the residuals of the model's predictions from the true sine.
sparse_checkpoint_updates = []
sparse_signal_errors = []

# For each snapshot of the model during training, calculate the RMSE against the true sine values and store the results.
for snapshot in sparse_snapshots:

    sparse_checkpoint_updates.append(snapshot["updates"])
    signal_error = calculate_rmse(snapshot["curve_predictions"], y_curve_true)
    sparse_signal_errors.append(signal_error)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

draw_training_and_signal_errors(axis=axes[0], losses= ## FINISH_ME ##,  # Use the sparse_loss_history as the losses
                                saved_updates=sparse_checkpoint_updates, signal_errors=sparse_signal_errors)

draw_waveform_residuals(axis=axes[1], observed_x=x_sparse, noise= ## FINISH_ME ##,  # Use the sparse_noise as the measurement noise
                        curve_x=x_curve, truth=y_curve_true, snapshots=sparse_snapshots)
plt.show()

# Print a summary of the training loss and RMSE against the true sine for each snapshot of the sparse model.
print("Stage    Training loss    RMSE from true sine (dense grid)")

# enumerate provides both a position (0 to 3) and its percentage label.
for index, percentage in enumerate([0, 25, 50, 100]):
    print(f"{percentage:3d}%     {sparse_snapshots[index]['loss']:.3e}        {sparse_signal_errors[index]:.4f}")


**Question:**

Why is the last model not necessarily the one we want to keep?


**Answer:**

` ## FINISH_ME ## `

## Part 6 — Fit a physical model to extract the frequency

**Question for this experiment:** does a known physical form let us recover the
waveform by fitting just its frequency?

Follow the same sequence: **prepare data → configure a fresh model → fit →
inspect results**. Here fitting searches candidate frequencies rather than
performing gradient updates.

This time physics supplies the model's shape. Assume **unit amplitude, zero phase
and zero offset**, and fit just the angular frequency:

$$y_{\mathrm{model}}(x;\omega)=\sin(\omega x).$$


**Exercise E11 — Fit a physical sinusoid**

Build this one-parameter model in the short steps below.
The search loop is supplied, just as the training loop was supplied in Section 1.8.


### 6.1 Prepare the inputs and targets

Use the **same 16 noisy observations as Part 5**, with the same $x$ coordinates
supplied directly to the model.

- $x$ is a dimensionless time coordinate in this experiment.
- $\omega$ is angular frequency per unit $x$.
- $\omega/(2\pi)$ is cycles per unit $x$, or hertz if $x$ is measured in seconds.

The clean sine values will be used only to assess the finished fit.


In [ ]:
# Give the existing observations names that make their role in this fit explicit.
frequency_inputs = x_sparse
frequency_targets = y_sparse

print("Input shape:", frequency_inputs.shape)  # Print the shape of the frequency_inputs array
print("Target shape:", ## FINISH_ME ##)  # Print the shape of the frequency_targets array
print("First three (coordinate, measured displacement) pairs:")
print(np.column_stack((frequency_inputs[:3, 0], frequency_targets[:3, 0])))


### 6.2 Choose a parameter and define the forward model

The MLP used a dictionary of weights and biases. This model needs a dictionary
with **one adjustable number**, `omega`. Choose 0.8 as an initial guess; we have
not fitted anything yet.

**Exercise E11A — Define the physical model**

Define the forward calculation. It should accept inputs and
parameters and return one prediction per input, just like the MLP's forward pass.


In [ ]:
# The angular frequency of the sine function is a single scalar parameter.
frequency_model = {"omega": 0.8}

# Define the forward pass for the sinusoid model.
def sinusoid_forward(inputs, model):
    """Predict a unit-amplitude, zero-phase sinusoid at the supplied coordinates."""

    # The angular frequency is extracted from the model dictionary.
    omega = model["omega"]
    predictions = np.sin(omega * inputs)

    return predictions

print("Initial angular frequency:", frequency_model["omega"])


### 6.3 Make a prediction and measure its error

Run the initial model forwards, then use **the same `loss_and_gradient` function**
as the MLP. This keeps the loss definition identical in the comparison.

**Exercise E11B — Evaluate the initial prediction**

Calculate the initial predictions and loss. The function also
returns prediction derivatives; our grid search below will not need them.


In [ ]:
# Compute the initial predictions
initial_frequency_predictions = sinusoid_forward(frequency_inputs, ### FINISH_ME ##)  # Use the frequency_model to compute the initial predictions

# Calculate the initial loss and gradient with respect to the predictions
initial_frequency_loss, initial_frequency_gradient = loss_and_gradient(initial_frequency_predictions, frequency_targets)
print(f"Initial half mean squared error: {initial_frequency_loss:.6f}")

# Calculate the initial RMSE against the noisy targets
initial_frequency_rmse = calculate_rmse(initial_frequency_predictions, frequency_targets)
print(f"Initial RMSE against noisy targets: {initial_frequency_rmse:.6f}")


### 6.4 Fit the parameter: a supplied search loop

With only one parameter, we can try a list of frequencies directly. This is a
**grid search**; minimising the squared-error loss makes it a **least-squares fit**.

For each candidate, the supplied loop:

1. Builds a trial parameter dictionary.
2. Calls your forward model on the observed inputs.
3. Calculates and stores its loss against the noisy targets.

The MLP used backpropagation to choose parameter changes. Here we compare trial
values, so no backward pass or gradient update is needed.

> **Search settings:** 0.5–1.5 assumes approximate prior knowledge of the frequency.
> Grid spacing controls search resolution; it is **not an uncertainty estimate**.


In [ ]:
# Evaluate candidates using the forward model and loss function.
candidate_omegas = np.linspace( ## FINISH_ME ## )  # Generate 2001 candidate angular frequencies from 0.5 to 1.5
frequency_losses = np.zeros(len(candidate_omegas))

for index in range(len(candidate_omegas)):

    # Each candidate defines a complete model with a different frequency.
    trial_model = {"omega": candidate_omegas[index]}

    # Compute the predictions and loss for this candidate frequency.
    trial_predictions = sinusoid_forward(frequency_inputs, trial_model)

    # Calculate the loss and gradient for the trial predictions against the frequency targets.
    trial_loss, trial_prediction_gradient = loss_and_gradient(trial_predictions, frequency_targets)
    frequency_losses[index] = trial_loss

print("Trial frequencies evaluated:", len(candidate_omegas))
print(f"Angular-frequency grid spacing: {candidate_omegas[1] - candidate_omegas[0]:.4f}")


### 6.5 Keep the best-fitting model

The loop has compared the candidates but has not changed `frequency_model`.
Now select the smallest recorded loss and store its frequency in that dictionary.
The model is then ready to make predictions at new coordinates.

**Exercise E11C — Keep the best-fitting frequency**

Use `np.argmin` to find the index of the smallest loss,
then replace the initial frequency with the fitted value.


In [ ]:
# Find the best-fitting frequency.
# np.argmin returns the index of the smallest value in the frequency_losses array.
# This index corresponds to the best-fitting angular frequency.
best_index = int(## FINISH_ME ##)  # Find the index of the minimum loss in frequency_losses
omega_fit = candidate_omegas[best_index]

frequency_model["omega"] = omega_fit
frequency_fit = omega_fit / (2.0 * np.pi)

# Report the results of the frequency search.
print(f"Fitted angular frequency: {omega_fit:.4f} per unit x")
print(f"Fitted frequency: {frequency_fit:.6f} cycles per unit x")
print(f"Training loss: {initial_frequency_loss:.6f} -> {frequency_losses[best_index]:.6f}")


### 6.6 Evaluate the fitted model

Use the fitted parameter in a forward pass over the dense plotting coordinates,
just as we did when inspecting the trained MLP.

Keep two questions separate:

- **Training RMSE:** how well does the model match the noisy observations?
- **True-sine RMSE:** how well does it recover the clean waveform across the plotting grid?

Only now use the synthetic truth to assess the fit. It did not select the frequency.


In [ ]:
# Compare the fitted sinusoid with the true sine function and the network's predictions.
physics_curve = sinusoid_forward(x_curve, frequency_model)
physics_predictions = sinusoid_forward(frequency_inputs, frequency_model)

# Calculate the RMSE of the fitted sinusoid against both the noisy targets and the true sine values.
physics_training_rmse = calculate_rmse(physics_predictions, frequency_targets)
physics_truth_rmse = calculate_rmse(physics_curve, y_curve_true)

# Assess the trained MLP against the same observations.
network_training_rmse = calculate_rmse(sparse_snapshots[-1]["training_predictions"], y_sparse)
network_parameter_count = SPARSE_HIDDEN_SIZE ** 2 + 4 * SPARSE_HIDDEN_SIZE + 1

# Both models are assessed against the same noisy targets and true-sine grid.
print(f"{'Model':<28}{'RMSE: noisy targets':>22}{'RMSE: true sine':>22}")
print(f"{'Sinusoid (1 parameter)':<28}{physics_training_rmse:>22.6f}{physics_truth_rmse:>22.6f}")

# The MLP has many more parameters than the sinusoid, but it is still assessed against the same noisy targets and true-sine grid.
network_label = f"MLP ({network_parameter_count} parameters)"
print(f"{network_label:<28}{network_training_rmse:>22.6f}{sparse_signal_errors[-1]:>22.6f}")


### 6.7 Inspect the result

| Panel | What to inspect |
|---|---|
| **Left: loss versus frequency** | The chosen parameter sits at the lowest recorded loss |
| **Right: waveforms** | Compare the physical model, overtrained MLP and clean reference |

The model-building steps are familiar. What changed is the set of shapes the
model can represent and the method used to choose its parameters.


In [ ]:
# Plot the loss landscape and the fitted waveforms for the sinusoid and the network.
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

def plot_loss_landscape(axis, candidate_omegas, frequency_losses, omega_fit):
    axis.set_title("Loss Landscape")

    # Plot the loss landscape for the sinusoid model as a function of trial angular frequencies.
    axis.plot(candidate_omegas, frequency_losses, color="tab:blue", label="Half MSE against noisy targets")

    # Mark the best-fitting angular frequency with a vertical dashed line and label it with its value.
    axis.axvline(omega_fit, color="tab:green", linestyle="--", label=f"Best omega = {omega_fit:.4f}")

    # Set the labels, title, and legend for the plot.
    axis.set(xlabel="Trial angular frequency", ylabel="Half MSE against noisy targets", title="Fit one physical parameter")
    axis.legend(fontsize=9)

def plot_noisy_observations_and_fitted_waveforms(axis, x_sparse, y_sparse, sparse_snapshots, physics_curve, y_curve_true):
    axis.set_title("Fitted waveforms")

    # Plot the noisy observations for the sparse dataset as blue dots.
    axis.scatter(x_sparse[:, 0], y_sparse[:, 0], s=22, color=## FINISH_ME ##, label="Same noisy observations")

    # Plot the fitted waveform for the overtrained network as an orange line.
    axis.plot(x_curve[:, 0], sparse_snapshots[-1]["curve_predictions"][:, 0], color=## FINISH_ME ##, label="Overtrained network")

    # Plot the fitted sinusoid for the best-fitting angular frequency as a green line.
    axis.plot(x_curve[:, 0], physics_curve[:, 0], color=## FINISH_ME ##, linewidth=2, label="Fitted sinusoid")

    # Plot the true sine function as a black dashed line.
    axis.plot(x_curve[:, 0], y_curve_true[:, 0], "k--", linewidth=1, label="True sine", color=## FINISH_ME ##)

    # Set the labels, title, and legend for the plot.
    axis.set(xlabel="Coordinate x", ylabel="Displacement", title="Compare the inferred waveforms")
    axis.legend(fontsize=8)

plot_loss_landscape(axis=axes[0], candidate_omegas=candidate_omegas,
                    frequency_losses=frequency_losses, omega_fit=omega_fit)

plot_noisy_observations_and_fitted_waveforms(axis=axes[1], x_sparse=x_sparse, y_sparse=y_sparse,
                                            sparse_snapshots=sparse_snapshots, physics_curve=physics_curve,
                                            y_curve_true=y_curve_true)

plt.show()


**Question:**

Why does the physics fit recover a better waveform despite its higher training loss? What assumption makes this possible?


**Answer:**

` ## FINISH_ME ## `